In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from ObjectSelectionHelper import ObjectSelectionHelper, make_lvec_M, make_lvec_E

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8f2e530
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8f59f70


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# prod = False
prod = True
no_rvec = True
write_outputs = False
# write_outputs = True
# plot_dir_postfix = "-new-cuts"
dataset_path = "data/datasets/pre-selected/checked-test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/selected-objects/test"
# output_meta_path = "data/datasets/selected-objects"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
# output_collections = [
#     "true_lep_lvec", "true_nu_lvec", "true_quark1_lvec", "true_quark2_lvec",
#     "iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec",
#     ]
# true lvecs do not exist in every df so cannot be explicitly requested...
# urgh but empty snapshots are also not allowed
# output_collections = r"(true_\w+_lvec)|(true_lep_charge)|(true_beam_\w+_lvec)|(iso_lep_charge)|(iso_lep_lvec)|(iso_lep_\w+_lvec)|(nu_lvec)|(R2Jet_sel1_lvec)|(R2Jet_sel2_lvec)|(R2Jet1_lvec)|(R2Jet2_lvec)|(iso_lep_mlvec)|(nu_mlvec)|(R2Jet_sel1_mlvec)|(R2Jet_sel2_mlvec)"
# plot_dir = f"plots/pre-selection/test{plot_dir_postfix}"
plot_dir=None
if prod:
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs-min-aa-min-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/min-higgs.json"
    dataset_path = "data/datasets/pre-selected/checked-signal-only.json"
    # output_path = "root://eospublic.cern.ch//eos/experiment/clicdp/data/user/l/lreichen/snapshots3/min-higgs-d"
    # output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/selected-objects/signal-only"
    # output_meta_path = "data/datasets/selected-objects"
    # output_meta = f"{output_meta_path}/signal-only.json"
    # checked_output_meta = f"{output_meta_path}/checked-signal-only.json"
    # plot_dir = "plots/object-selection/signal-only"

In [4]:
ROOT.EnableImplicitMT(n_threads)

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ObjectSelectionHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xcb12b30


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]
background_categories = [cat for cat in analysis.get_categories() if cat not in signal_category]
print(signal_category)
print(background_categories)

True
['4f_sw_sl_signal']
['4f_sl_bkg']


In [8]:
ROOT.gStyle.SetOptStat(1)

In [9]:
analysis.define_truth_objects(signal_category)

In [10]:
analysis.define_only_on(signal_category, "true_nu_Pz", "true_nu_lvec.Pz()")

In [11]:
# make nominal iso lep
# try to make iso lep + brems, try to make cheated brems
analysis.Define("iso_lep_idx", "IsolatedElectrons_objIdx.index[0]")
analysis.Define("iso_lep_charge", "PandoraPFOs.charge[iso_lep_idx]")
analysis.Define("iso_lep_lvec", "ROOT::Math::PxPyPzEVector(PandoraPFOs.momentum.x[iso_lep_idx], PandoraPFOs.momentum.y[iso_lep_idx], PandoraPFOs.momentum.z[iso_lep_idx], PandoraPFOs.energy[iso_lep_idx])")
analysis.Define("iso_lep_E", "iso_lep_lvec.energy()")
analysis.Define("iso_lep_M", "iso_lep_lvec.M()")

analysis.Define("n_iso_gammas", "IsolatedPhotons_objIdx.index.size()")
analysis.Define("iso_gamma_idx", "IsolatedPhotons_objIdx.index[0]")
analysis.Define("iso_gamma_lvec", "ROOT::Math::PxPyPzEVector(PandoraPFOs.momentum.x[iso_gamma_idx], PandoraPFOs.momentum.y[iso_gamma_idx], PandoraPFOs.momentum.z[iso_gamma_idx], PandoraPFOs.energy[iso_gamma_idx])")

# analysis.Define("iso_lep_gamma_deltaR", "ROOT::Math::VectorUtil::DeltaR(iso_lep_lvec, iso_gamma_lvec)")
analysis.Define("iso_lep_gamma_angle", "ROOT::Math::VectorUtil::Angle(iso_lep_lvec, iso_gamma_lvec)")
analysis.Define("iso_lep_gamma_close", "iso_lep_gamma_angle > 0. && iso_lep_gamma_angle < 0.01")

analysis.Define("iso_lep_gamma_lvec", "n_iso_gammas > 0 && iso_lep_gamma_close ? iso_lep_lvec + iso_gamma_lvec : iso_lep_lvec")

In [12]:
# brems recovery a la Sang Hyun
analysis.Define("PFO_lvecs", "Construct<ROOT::Math::PxPyPzEVector>(PandoraPFOs.momentum.x, PandoraPFOs.momentum.y, PandoraPFOs.momentum.z, PandoraPFOs.energy)")
analysis.Define("PFO_etas", "return Map(PFO_lvecs, [] (const auto &el) {return el.eta();} )")
analysis.Define("PFO_phis", "return Map(PFO_lvecs, [] (const auto &el) {return el.phi();} )")
analysis.Define("brems_PFOs", "PandoraPFOs.PDG == 22 && abs(PFO_etas - iso_lep_lvec.eta()) <= 0.005 &&  abs(PFO_phis - iso_lep_lvec.phi()) <= 0.05")

analysis.Define("iso_lep_brems_lvec", "iso_lep_lvec + Sum(PFO_lvecs[brems_PFOs], ROOT::Math::PxPyPzEVector())")
# TODO: do something with this!
# I.e. remove identified brems from jets
# might also be worth to consider to add other isolated particles back to jets somehow...

In [13]:
ROOT.gInterpreter.Declare("#include \"analyzers.h\"")
# cheated brems finding from MC truth
analysis.define_only_on(signal_category, "true_brems_indices", "find_brems(true_lep_idx, MCParticlesSkimmed.daughters_begin, MCParticlesSkimmed.daughters_end, _MCParticlesSkimmed_daughters.index, MCParticlesSkimmed.PDG)")
analysis.book_histogram_1D("true_brems_indices", "true_brems_indices", ("", "", 190, 10., 200.), categories=signal_category)

In [14]:

analysis.define_only_on(signal_category, "mcp_brems_mask", "subset_to_mask(MCParticlesSkimmed.PDG.size(), true_brems_indices)")
analysis.define_only_on(signal_category, "pfo_brems_mask", "mcp_mask_to_pfo_mask(mcp_brems_mask, PandoraPFOs, _RecoMCTruthLink_from, _RecoMCTruthLink_to, RecoMCTruthLink_weight)")

analysis.define_only_on(signal_category, "brems_PFO_PDG", "PandoraPFOs.PDG[pfo_brems_mask]")
analysis.define_only_on(signal_category, "brems_PFO_energy", "PandoraPFOs.energy[pfo_brems_mask]")
analysis.define_only_on(signal_category, "n_brems_PFOs", "Sum(pfo_brems_mask, 0)")

analysis.book_histogram_1D("n_brems_PFOs", "n_brems_PFOs", ("", ";N_{brems PFOs};Events", 25, 0., 25.), categories=signal_category)
analysis.book_histogram_1D("brems_PFO_PDG", "brems_PFO_PDG", ("", ";Brems PFO PDG;Events", 1000, -500, 500), categories=signal_category)

analysis.define_only_on(signal_category, "cheated_brems_lvec", "PFO_lvecs[pfo_brems_mask][0]")
analysis.define_only_on(signal_category, "cheated_brems_lvecs_E", "cheated_brems_lvec.E()")
analysis.define_only_on(signal_category, "cheated_brems_lvecs_P", "cheated_brems_lvec.P()")
analysis.define_only_on(signal_category, "cheated_brems_lvecs_theta", "cheated_brems_lvec.Theta()")
analysis.define_only_on(signal_category, "cheated_brems_lvecs_phi", "cheated_brems_lvec.Phi()")
analysis.book_histogram_1D("cheated_brems_lvecs_E", "cheated_brems_lvecs_E", ("", ";E_{brems PFO} [GeV];Events", 100, 0., 100.), categories=signal_category)
analysis.book_histogram_1D("brems_PFO_energy", "brems_PFO_energy", ("", ";E_{brems PFO} [GeV];Events", 100, 0., 25.), categories=signal_category)
analysis.book_histogram_1D("cheated_brems_lvecs_P", "cheated_brems_lvecs_P", ("", ";P_{brems PFO} [GeV];Events", 100, 0., 100.), categories=signal_category)

analysis.define_only_on(signal_category, "iso_lep_cheated_brems_lvec", "iso_lep_lvec + cheated_brems_lvec")
# analysis.Define("cheat_clean_R2Jets", "remove_constituents(Refined2Jets, _Refined2Jets_particles, PandoraPFOs, pfo_ovl_mask)")
# analysis.Define("cheat_clean_R2Jet_lvecs", "return Map(cheat_clean_R2Jets, [] (const auto& el) {return ROOT::Math::PxPyPzEVector(el.momentum.x, el.momentum.y, el.momentum.z, el.energy);})")

In [15]:
analysis.define_only_on(signal_category, "true_lep_E", "true_lep_lvec.energy()")

analysis.define_deltas("iso_lep", "iso_lep_lvec", "true_lep_lvec", signal_category)
analysis.define_deltas("iso_lep_gamma", "iso_lep_gamma_lvec", "true_lep_lvec", signal_category)
analysis.define_deltas("iso_lep_brems", "iso_lep_brems_lvec", "true_lep_lvec", signal_category)
analysis.define_deltas("iso_lep_cheated_brems", "iso_lep_cheated_brems_lvec", "true_lep_lvec", signal_category)

analysis.define_only_on(signal_category, "iso_lep_delta_phi_q", "iso_lep_delta_phi * true_lep_charge")
analysis.define_only_on(signal_category, "iso_lep_gamma_delta_phi_q", "iso_lep_gamma_delta_phi * true_lep_charge")
analysis.define_only_on(signal_category, "iso_lep_brems_delta_phi_q", "iso_lep_brems_delta_phi * true_lep_charge")


In [16]:
analysis.book_histogram_1D("true_lep_E", "true_lep_E", ("", "true lep E", 125, 0., 125.), categories=signal_category)

for name in ["iso_lep", "iso_lep_gamma", "iso_lep_brems", "iso_lep_cheated_brems"]:
    analysis.book_histogram_1D(f"{name}_delta_E", f"{name}_delta_E", ("", ";#Delta E [GeV]", 150, -10., 5.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_P", f"{name}_delta_P", ("", ";#Delta P [GeV]", 150, -10., 5.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_theta", f"{name}_delta_theta", ("", ";#Delta #theta [rad]", 150, -0.0005, 0.0005), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_phi", f"{name}_delta_phi", ("", ";#Delta #phi [rad]", 150, -0.0005, 0.0005), categories=signal_category)

analysis.book_histogram_1D("iso_lep_delta_phi_q", "iso_lep_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)
analysis.book_histogram_1D("iso_lep_gamma_delta_phi_q", "iso_lep_gamma_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)
analysis.book_histogram_1D("iso_lep_brems_delta_phi_q", "iso_lep_brems_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)

# analysis.book_histogram_1D("iso_lep_gamma_deltaR", "iso_lep_gamma_deltaR", ("", ";#Delta R lep gamma", 300, 0., 0.02), categories=signal_category)
analysis.book_histogram_1D("iso_lep_gamma_angle", "iso_lep_gamma_angle", ("", ";Angle lep gamma", 300, 0., 0.02), categories=signal_category)

In [17]:
%%time
analysis.run()

CPU times: user 5min 6s, sys: 36.8 s, total: 5min 43s
Wall time: 2min 46s


In [18]:
analysis.draw_summed_unscaled_histograms("true_brems_indices", signal_category[0])
analysis.draw_summed_unscaled_histograms("n_brems_PFOs", signal_category[0])
analysis.draw_summed_unscaled_histograms("brems_PFO_PDG", signal_category[0])
analysis.draw_summed_unscaled_histograms("cheated_brems_lvecs_E", signal_category[0])
analysis.draw_summed_unscaled_histograms("cheated_brems_lvecs_P", signal_category[0])
analysis.draw_summed_unscaled_histograms("brems_PFO_energy", signal_category[0])

In [19]:
# analysis.compare_summed_histograms_unscaled(["iso_lep_delta_E"], signal_category[0], legend_remove_suffix="_delta_E", plot_dir=plot_dir)
# analysis.compare_summed_histograms_unscaled(["iso_lep_delta_P"], signal_category[0], legend_remove_suffix="_delta_P", plot_dir=plot_dir)
# analysis.compare_summed_histograms_unscaled(["iso_lep_delta_theta"], signal_category[0], legend_remove_suffix="_delta_theta", plot_dir=plot_dir)
# analysis.compare_summed_histograms_unscaled(["iso_lep_delta_phi"], signal_category[0], legend_remove_suffix="_delta_phi", plot_dir=plot_dir)
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_E", "iso_lep_gamma_delta_E", "iso_lep_brems_delta_E", "iso_lep_cheated_brems_delta_E"], signal_category[0], legend_remove_suffix="_delta_E", plot_dir=plot_dir)
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_P", "iso_lep_gamma_delta_P", "iso_lep_brems_delta_P", "iso_lep_cheated_brems_delta_P"], signal_category[0], legend_remove_suffix="_delta_P", plot_dir=plot_dir)
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_theta", "iso_lep_gamma_delta_theta", "iso_lep_brems_delta_theta", "iso_lep_cheated_brems_delta_theta"], signal_category[0], legend_remove_suffix="_delta_theta", plot_dir=plot_dir)
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_phi", "iso_lep_gamma_delta_phi", "iso_lep_brems_delta_phi", "iso_lep_cheated_brems_delta_phi"], signal_category[0], legend_remove_suffix="_delta_phi", plot_dir=plot_dir)

h1 title: #Delta E [GeV]
h1 title: #Delta P [GeV]
h1 title: #Delta #theta [rad]
h1 title: #Delta #phi [rad]
